# Lesson 02 — Tracking Objects by Color

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Demo on static image — in .py file do this on webcam frames
img = cv2.imread('sample.jpg')
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Track a colored ball or object
lower = np.array([20, 100, 100])  # adjust for your object's HSV
upper = np.array([30, 255, 255])

mask = cv2.inRange(hsv, lower, upper)
# Clean up
mask = cv2.erode( mask, None, iterations=2)
mask = cv2.dilate(mask, None, iterations=2)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

vis = img.copy()
if contours:
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) > 200:
        (x,y), radius = cv2.minEnclosingCircle(c)
        M  = cv2.moments(c)
        cx = int(M['m10']/(M['m00']+1e-6))
        cy = int(M['m01']/(M['m00']+1e-6))
        cv2.circle(vis, (int(x),int(y)), int(radius), (0,255,0), 2)
        cv2.circle(vis, (cx,cy), 5, (0,0,255), -1)
        cv2.putText(vis, f'({cx},{cy})', (cx+10,cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        print(f"Object center: ({cx},{cy}), radius: {radius:.0f}px")

plt.subplot(1,2,1); plt.imshow(mask, cmap='gray'); plt.title('Color mask'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.title('Tracked object'); plt.axis('off')
plt.show()

## Key Takeaway
Color tracking pipeline: HSV → inRange → erode/dilate → find largest contour → get centroid.
In a video loop, the centroid gives you real-time position tracking.